# 09a · Datación PCA-MM12 sin `torch` (vigente: niveles, 16 series)

**Objetivo.** Datación de regímenes por PCA del notebook 09 y del Rmd final (sección 6), sin
depender de `torch`: PCA de la matriz de correlaciones con la regla de signo de Rcamiz; scores
suavizados con una media móvil **causal** de 12 meses; partición óptima de Fisher de los scores
estandarizados, con regímenes de al menos 12 meses.

**Criterios fijados de antemano.**
- **Regla K** (D12): el menor $k$ cuya reducción relativa de la suma de cuadrados al pasar a $k+1$
  es menor que 10 % y, si ninguno lo es, la caída máxima de esa reducción.
- **maxcl = 10** es la especificación principal (D17.1); maxcl = 15 y 20 se informan como
  sensibilidad. La columna `n_segmentos_en_minimo` cuenta los tramos de exactamente 12 meses y
  `segmentos_en_minimo` marca (lectura literal de D17.1) las soluciones con **todos** los tramos en
  el mínimo.
- **d = 3** (D9) es la dimensión principal; d = 2 y d = 4 son la sensibilidad de D12 (tarea D12).
- El CH se informa en sus dos variantes y no selecciona (D11).

**Parámetros** (`src/parametros.py`): por defecto `VIA = 'niveles'`, 16 series (D16) y salidas en
`data/results/final/`. Con `TESIS_VIA=ambas` y `TESIS_EXCLUIDAS=[]` reproduce el antecedente con
17 series y dos vías (prueba de no regresión, `herramientas/no_regresion_dinamica.py`).

**Semillas.** Ninguna propia: las réplicas bootstrap son las del Rmd (`R_bootstrap_indices_L24.csv`,
semilla 2026). **Orden:** el Rmd final se renderiza antes que este notebook; si sus `R_*` existen,
este notebook los compara con los suyos y falla si difieren.

**Salidas:** `09a_regimenes_pca.csv`, `09a_estabilidad_cortes.csv`, `09a_datacion_criterios.csv`
(d = 3) y `09a_datacion_dimension.csv` (d ∈ {2, 3, 4}).

In [1]:
# ---- Configuración y parámetros (adenda 3b, §3) ------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src.parametros import leer_parametros
from src.datos import cargar_vias, meta_vigente

PARAM = leer_parametros(RAIZ)
VIA, EXCLUIDAS, DIR, DIR_R = PARAM['VIA'], PARAM['EXCLUIDAS'], PARAM['DIR'], PARAM['DIR_R']
ANTECEDENTE = len(EXCLUIDAS) == 0          # 17 series: modo de no regresión
X, Xt = cargar_vias(RAIZ, excluidas=EXCLUIDAS)
META = meta_vigente(EXCLUIDAS)
P = X.shape[1]
TODAS = {'niveles': X, 'transformadas': Xt}
VIAS = {v: (TODAS[v], DIR[v], DIR_R[v]) for v in PARAM['VIAS_ACTIVAS']}
print('VIA =', VIA, '| EXCLUIDAS =', EXCLUIDAS, '| p =', P, '| CARPETA =', PARAM['CARPETA'])
for via, (M, d, dr) in VIAS.items():
    print(via, M.shape, M.index[0], '→', M.index[-1], '| salidas:', d.relative_to(RAIZ) if d.is_relative_to(RAIZ) else d)

from src import particion as pt
from src.datos import CREDITOS
from src.episodios import cargar_episodios
from src.remuestreo import leer_inicios_bootstrap, bootstrap_pca

SEMILLA = None            # sin sorteos propios: réplicas del Rmd
L_MM = 12                 # media móvil causal (meses)
N_MIN = 12                # longitud mínima de un régimen (meses)
UMBRAL = 0.10             # regla K, paso 1 (D12)
D = 3                     # D9: d = 3
DIMENSIONES = (2, 3, 4)   # D12: sensibilidad de la dimensión
MAXCL_PRINCIPAL = 10      # D17.1
MAXCL_GRILLA = (10, 15, 20)
B_CORTES, L_BOOT, TOL_MESES = 200, 24, 3
EPISODIOS = cargar_episodios(RAIZ)

VIA = niveles | EXCLUIDAS = ('Liquidez M2',) | p = 16 | CARPETA = /home/claude/tesis-modelamiento-matematico/data/results/final
niveles (228, 16) 2007-01 → 2025-12 | salidas: data/results/final


## 1. Datación principal (d = 3, maxcl = 10, regla K)

Fases (regla del 09): **expansión** si el promedio del componente de crédito (media de los tres
créditos estandarizados en toda la muestra) es mayor que 0,25, **contracción** si es menor que
−0,25 y **transición** si está entre ambos o si el régimen dura menos de 18 meses. *Episodios
contenidos:* los del archivo único (D7, D9) que se superponen con el régimen (un episodio puntual
ocupa solo su mes de inicio).

In [2]:
def suavizar(M, d=D):
    # Scores PC1..PCd con la regla de Rcamiz y media móvil causal de 12 meses (sin los NaN iniciales)
    val, V, S = pt.scores_pca_rcamiz(M.to_numpy())
    Sm = np.column_stack([pt.media_movil_causal(S[:, k], L_MM) for k in range(d)])[L_MM - 1:]
    return val, V, Sm, list(M.index[L_MM - 1:])


def episodios_contenidos(ini, fin):
    a, b = pd.Timestamp(ini + '-01'), pd.Timestamp(fin + '-01')
    return ', '.join(r.nombre for r in EPISODIOS.itertuples() if r.inicio_mes <= b and r.fin_efectivo >= a)


def fases(M, cortes, fechas_seg, umbral=0.25, dur_min=18):
    Z = pt.estandarizar_r(M.to_numpy())
    pos = [list(M.columns).index(c) for c in CREDITOS]
    comp = (np.sum(Z[:, pos].astype(np.longdouble), axis=1) / len(pos)).astype(float)   # rowMeans
    fechas = list(M.index)
    bordes = [0] + list(cortes) + [len(fechas_seg)]
    filas = []
    for a, b in zip(bordes[:-1], bordes[1:]):
        ini, fin = fechas_seg[a], fechas_seg[b - 1]
        sel = np.array([ini <= f <= fin for f in fechas])
        m, dur = float(np.mean(comp[sel])), int(sel.sum())
        fase = 'Transición' if dur < dur_min or abs(m) <= umbral else ('Expansión' if m > 0 else 'Contracción')
        filas.append({'inicio': ini, 'fin': fin, 'meses': dur, 'componente_credito_medio': round(m, 2), 'fase': fase,
                      'episodios_contenidos': episodios_contenidos(ini, fin)})
    return pd.DataFrame(filas)


datacion = {}
for via, (M, d, dr) in VIAS.items():
    val, V, Sm, fz = suavizar(M)
    dat = pt.datar(Sm, k_max=MAXCL_PRINCIPAL, n_min=N_MIN, umbral=UMBRAL)
    cortes = dat['res'][dat['k_codo']]['cortes']
    reg = fases(M, cortes, fz)
    reg.to_csv(d / '09a_regimenes_pca.csv', index=False)
    ref09 = dr / '09_regimenes_pca.csv'
    if ANTECEDENTE and ref09.exists():
        # No regresión (17 series): fechas, k, duración y fase exactos del notebook 09
        ref = pd.read_csv(ref09)
        assert len(ref) == len(reg) == dat['k_codo'], (via, len(ref), len(reg))
        for c in ['inicio', 'fin', 'meses', 'fase']:
            assert (ref[c].astype(str).to_numpy() == reg[c].astype(str).to_numpy()).all(), (via, c)
        assert np.max(np.abs(ref.componente_credito_medio - reg.componente_credito_medio)) < 0.0051
        print(f'{via}: reproduce {ref09.name} del antecedente')
    datacion[via] = {'val': val, 'V': V, 'Sm': Sm, 'fz': fz, 'dat': dat, 'cortes': cortes}
    print(f"{via}: k = {dat['k_codo']} (regla K, paso «{dat['regla_codo']}», maxcl = {MAXCL_PRINCIPAL})")
    display(reg)

niveles: k = 6 (regla K, paso «caida», maxcl = 10)


,inicio,fin,meses,componente_credito_medio,fase,episodios_contenidos
0,2007-12,2008-11,12,-1.45,Transición,Crisis financiera global
1,2008-12,2010-02,15,-1.30,Transición,Crisis financiera global
2,2010-03,2014-03,49,-0.79,Contracción,Taper tantrum
3,2014-04,2020-05,74,0.17,Transición,"Colapso de las materias primas, Kuczynski / Od..."
4,2020-06,2022-10,29,0.97,Expansión,
5,2022-11,2025-12,38,1.39,Expansión,


## 2. Estabilidad de los cortes

Para cada réplica $b$ (las 200 primeras de `R_bootstrap_indices_L24.csv`), las cargas PC1–PC4 se
emparejan con las originales como en el bootstrap del 05; los scores se recalculan con
$V_b = C_b / \sqrt{\lambda_b}$ sobre las series estandarizadas, se suavizan y se parten en el mismo
número de regímenes. Se cuenta la proporción de réplicas con un corte a ±3 meses de cada corte
original. *Supuesto:* el bootstrap por bloques supone dependencia débil; en niveles no se cumple, y
la cifra mide estabilidad descriptiva de las fechas, no incertidumbre muestral.

In [3]:
estabilidad = {}
for via, (M, d, dr) in VIAS.items():
    r = datacion[via]
    cargas_ref = r['V'] * np.sqrt(r['val'])
    inicios = leer_inicios_bootstrap(dr / 'R_bootstrap_indices_L24.csv')[:B_CORTES]
    bt = bootstrap_pca(M.to_numpy(), cargas_ref, inicios, L_BOOT, K=4)
    Z = pt.estandarizar_r(M.to_numpy())
    k_ref = r['dat']['k_codo']
    frec, _ = pt.estabilidad_cortes(Z, bt['cargas'], bt['valores_emparejados'], r['cortes'], k_ref,
                                    ventana=L_MM, n_min=N_MIN, tolerancia=TOL_MESES, d=D)
    tab = pd.DataFrame({'corte': pt.fechas_corte(r['cortes'], r['fz']), 'frecuencia': frec, 'k_ref': k_ref,
                        'replicas': B_CORTES, 'L': L_BOOT, 'tolerancia_meses': TOL_MESES})
    tab.to_csv(d / '09a_estabilidad_cortes.csv', index=False)
    estabilidad[via] = tab
    f_r = dr / 'R_estabilidad_cortes.csv'
    if f_r.exists():
        ref = pd.read_csv(f_r)
        assert list(ref.corte) == list(tab.corte), via
        dif = float(np.max(np.abs(ref.frecuencia.to_numpy() - tab.frecuencia.to_numpy())))
        assert dif < 1e-12, (via, dif)
        print(f'{via}: coincide con {f_r.name} (dif. máx. {dif:.1e})')
    else:
        print(f'{via}: aún no existe {f_r.name} (ejecutar el Rmd)')
    display(tab)

niveles: coincide con R_estabilidad_cortes.csv (dif. máx. 0.0e+00)


,corte,frecuencia,k_ref,replicas,L,tolerancia_meses
0,2008-12,0.855,6,200,24,3
1,2010-03,0.975,6,200,24,3
2,2014-04,0.400,6,200,24,3
3,2020-06,0.955,6,200,24,3
4,2022-11,1.000,6,200,24,3


## 3. Grilla maxcl × criterio (d = 3)

`k_max_efectivo` es el mayor $k$ factible con tramos de al menos 12 meses (18 con maxcl = 20).
`k_en_tope` indica si el criterio eligió ese máximo. Las columnas nuevas de D17.1
(`n_segmentos_en_minimo`, `segmentos_en_minimo`) van al final, de modo que las columnas del
antecedente no cambian.

In [4]:
COLS_ANTECEDENTE = ['maxcl', 'k_max_efectivo', 'criterio', 'k', 'k_en_tope', 'regla_k', 'cortes']
COLS_NUEVAS = ['n_segmentos_en_minimo', 'segmentos_en_minimo']


def clave_grilla(t, cols):
    return [' '.join(str(x).upper() if isinstance(x, (bool, np.bool_)) else str(x) for x in fila)
            for fila in t[cols].itertuples(index=False)]


def comparar_con_r(tab, f_r, cols):
    ref = pd.read_csv(f_r, keep_default_na=False)
    py = tab.copy()
    for c in ['regla_k', 'cortes']:
        py[c] = py[c].fillna('')
    cols = [c for c in cols if c in ref.columns]
    ref_b = ref.copy()
    for c in cols:
        if ref_b[c].dtype == object and set(ref_b[c]) <= {'TRUE', 'FALSE'}:
            ref_b[c] = ref_b[c] == 'TRUE'
    n_dif = sum(a != b for a, b in zip(clave_grilla(ref_b, cols), clave_grilla(py, cols)))
    assert len(ref) == len(py) and n_dif == 0, (f_r.name, n_dif)
    return len(cols)


grillas = {}
for via, (M, d, dr) in VIAS.items():
    r = datacion[via]
    g = pt.grilla_datacion(r['Sm'], r['fz'], maxcls=MAXCL_GRILLA, n_min=N_MIN, umbral=UMBRAL)
    # Con maxcl = 10 (D17.1), la regla K reproduce la datación principal de la sección 1
    fila = g[(g.maxcl == MAXCL_PRINCIPAL) & (g.criterio == 'regla K (D12)')].iloc[0]
    assert fila.k == r['dat']['k_codo'] and fila.cortes == ' '.join(pt.fechas_corte(r['cortes'], r['fz']))
    g = g[COLS_ANTECEDENTE + COLS_NUEVAS]
    g.to_csv(d / '09a_datacion_criterios.csv', index=False)
    grillas[via] = g
    f_r = dr / 'R_datacion_criterios.csv'
    if f_r.exists():
        nc = comparar_con_r(pd.read_csv(d / '09a_datacion_criterios.csv', keep_default_na=False), f_r, COLS_ANTECEDENTE + COLS_NUEVAS)
        print(f'{via}: coincide con {f_r.name} ({len(g)} filas, {nc} columnas)')
    else:
        print(f'{via}: aún no existe {f_r.name} (ejecutar el Rmd)')
    display(g)

niveles: coincide con R_datacion_criterios.csv (12 filas, 9 columnas)


,maxcl,k_max_efectivo,criterio,k,k_en_tope,regla_k,cortes,n_segmentos_en_minimo,segmentos_en_minimo
0,10,10,codo (caída máxima),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11,1,False
1,10,10,Calinski-Harabász k gl (aede),10,True,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...,1,False
2,10,10,Calinski-Harabász k-1 gl (habitual),10,True,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...,1,False
3,10,10,regla K (D12),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11,1,False
4,15,15,codo (caída máxima),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11,1,False
5,15,15,Calinski-Harabász k gl (aede),12,False,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...,2,False
6,15,15,Calinski-Harabász k-1 gl (habitual),11,False,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...,2,False
7,15,15,regla K (D12),11,False,umbral,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...,2,False
8,20,18,codo (caída máxima),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11,1,False
9,20,18,Calinski-Harabász k gl (aede),12,False,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...,2,False


## 4. Sensibilidad de la dimensión: d ∈ {2, 3, 4} (D12)

La misma grilla con los scores PC1–PC$d$. La fila de d = 3 reproduce la sección 3.

In [5]:
dimension = {}
for via, (M, d, dr) in VIAS.items():
    partes = []
    for dd in DIMENSIONES:
        _, _, Sm, fz = suavizar(M, dd)
        partes.append(pt.grilla_datacion(Sm, fz, maxcls=MAXCL_GRILLA, n_min=N_MIN, umbral=UMBRAL).assign(d=dd))
    t = pd.concat(partes, ignore_index=True)[['d'] + COLS_ANTECEDENTE + COLS_NUEVAS]
    s3 = t[t.d == D].drop(columns='d').reset_index(drop=True)
    assert s3.astype(str).equals(grillas[via].reset_index(drop=True).astype(str)), via
    t.to_csv(d / '09a_datacion_dimension.csv', index=False)
    dimension[via] = t
    f_r = dr / 'R_datacion_dimension.csv'
    if f_r.exists():
        nc = comparar_con_r(pd.read_csv(d / '09a_datacion_dimension.csv', keep_default_na=False), f_r, ['d'] + COLS_ANTECEDENTE + COLS_NUEVAS)
        print(f'{via}: coincide con {f_r.name} ({len(t)} filas, {nc} columnas)')
    else:
        print(f'{via}: aún no existe {f_r.name} (ejecutar el Rmd)')
    display(t[t.criterio == 'regla K (D12)'])

niveles: coincide con R_datacion_dimension.csv (36 filas, 10 columnas)


,d,maxcl,k_max_efectivo,criterio,k,k_en_tope,regla_k,cortes,n_segmentos_en_minimo,segmentos_en_minimo
3,2,10,10,regla K (D12),6,False,caida,2010-01 2011-09 2018-02 2020-08 2022-09,0,False
7,2,15,15,regla K (D12),10,False,umbral,2010-01 2011-07 2013-12 2016-05 2018-05 2020-0...,1,False
11,2,20,18,regla K (D12),10,False,umbral,2010-01 2011-07 2013-12 2016-05 2018-05 2020-0...,1,False
15,3,10,10,regla K (D12),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11,1,False
19,3,15,15,regla K (D12),11,False,umbral,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...,2,False
23,3,20,18,regla K (D12),11,False,umbral,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...,2,False
27,4,10,10,regla K (D12),4,False,caida,2012-04 2020-05 2022-11,0,False
31,4,15,15,regla K (D12),13,False,umbral,2008-12 2010-01 2011-02 2012-10 2013-12 2015-0...,3,False
35,4,20,18,regla K (D12),13,False,umbral,2008-12 2010-01 2011-02 2012-10 2013-12 2015-0...,3,False


**Lectura.** Este notebook no interpreta los regímenes (regla 7). Toda lectura de un $k^*$ o de un
corte se hace contra el nulo de estructura constante (D7 del plan, D10 y D11), nunca contra los
episodios (regla 6).